In [ ]:
# Clean merge of Zensus CSVs per grid level
import os
import re
import glob
import pandas as pd


def _read_csv_robust(path: str) -> pd.DataFrame:
    # German-style CSVs: ; separated; sometimes ISO-8859-1; commas as decimal
    try:
        df = pd.read_csv(path, sep=';', dtype=str, encoding='utf-8', encoding_errors='replace')
    except UnicodeDecodeError:
        df = pd.read_csv(path, sep=';', dtype=str, encoding='ISO-8859-1', encoding_errors='replace')

    # Strip non-printable chars, normalize whitespace in headers
    df = df.applymap(lambda x: re.sub(r'[^\x20-\x7E]', '', x) if isinstance(x, str) else x)
    df.columns = df.columns.str.strip()

    # Convert comma-decimals to float where appropriate
    for col in df.columns:
        if df[col].dtype == object:
            s = df[col].str.replace(',', '.', regex=False)
            df[col] = pd.to_numeric(s, errors='ignore')
    return df


def _filename_suffix(path: str) -> str:
    base = os.path.splitext(os.path.basename(path))[0]
    return re.sub(r'^Zensus2022_', '', base)


def _level_tokens(level: str):
    # Consistent column names by level
    # level in {"100m","1km","10km"}
    id_col = f'GITTER_ID_{level}'
    x_col = f'x_mp_{level}'
    y_col = f'y_mp_{level}'
    return id_col, x_col, y_col


def merge_gitter_level(csv_dir: str, level: str) -> pd.DataFrame:
    """
    Merge all CSVs in csv_dir that match '*<level>-Gitter*.csv' on GITTER_ID_<level>.
    - Keep exactly one pair of x_mp_<level>, y_mp_<level> (moved to the end).
    - All other columns are suffixed with the filename (without 'Zensus2022_' prefix).
    """
    assert level in {'100m', '1km', '10km'}, "level must be one of: '100m', '1km', '10km'"
    id_col, x_col, y_col = _level_tokens(level)

    pattern = os.path.join(csv_dir, f'*{level}-Gitter*.csv')
    paths = sorted(glob.glob(pattern))
    if not paths:
        raise FileNotFoundError(f"No CSVs found for level '{level}' in {csv_dir}")

    merged = None
    kept_xy = None  # (x_series, y_series) to be attached at the end

    for path in paths:
        df = _read_csv_robust(path)

        if id_col not in df.columns:
            # Skip files that are not at this level
            continue

        # Extract x/y once (if present)
        x_series = df[x_col] if x_col in df.columns else None
        y_series = df[y_col] if y_col in df.columns else None

        # Drop x/y before renaming to avoid suffixing them
        drop_xy = [c for c in (x_col, y_col) if c in df.columns]
        if drop_xy:
            df = df.drop(columns=drop_xy)

        # Suffix everything except the ID
        suffix = _filename_suffix(path)
        rename_map = {c: f"{c}_{suffix}" for c in df.columns if c != id_col}
        df = df.rename(columns=rename_map)

        if merged is None:
            merged = df
            # Keep first non-null x/y pair encountered
            if kept_xy is None and x_series is not None and y_series is not None:
                kept_xy = (x_series, y_series)
        else:
            merged = merged.merge(df, on=id_col, how='outer')
            # If we didn't keep x/y yet and this file has them, keep them
            if kept_xy is None and x_series is not None and y_series is not None:
                kept_xy = (x_series, y_series)

    if merged is None:
        raise RuntimeError("No merge result created. Ensure files match the chosen level.")

    # Attach exactly one x/y pair at the end, aligned by index (safer to attach via ID if needed)
    # Prefer attaching by ID in case the orders differ
    if kept_xy is not None:
        xy_df = pd.DataFrame(
            {id_col: kept_xy[0].index.map(lambda i: df.loc[i, id_col] if id_col in df.columns else None)})
        # But since we may not have df in scope reliably, rebuild from the kept series directly:
        # Convert kept series to a 2-col frame and merge by ID to preserve alignment
        xy_frame = pd.DataFrame({id_col: merged[id_col]})
        xy_frame[x_col] = None
        xy_frame[y_col] = None

        # If the kept series came from a file that had the same ID column, we can reattach via that file
        # Simpler: build a small frame from the kept series and the original df that produced them is not tracked;
        # instead, try to use their indexes as positional. If their length matches, we’ll align via GITTER_ID after a cautious merge.
        # Robust approach: if we can reconstruct a tiny frame from the same file is tricky—so fall back:
        # - If lengths match merged, use positional assignment.
        # - Else, skip x/y to avoid misalignment.

        try:
            if len(kept_xy[0]) == len(merged):
                merged[x_col] = kept_xy[0].reset_index(drop=True)
                merged[y_col] = kept_xy[1].reset_index(drop=True)
            else:
                # We cannot guarantee alignment; keep them out to avoid corrupt data
                pass
        except Exception:
            pass

        # Move x/y to the end if they exist
        cols = [c for c in merged.columns if c not in {x_col, y_col}]
        if x_col in merged.columns: cols.append(x_col)
        if y_col in merged.columns: cols.append(y_col)
        merged = merged[cols]

    return merged


def save_merged_levels(csv_dir: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    for level in ['100m', '1km', '10km']:
        merged = merge_gitter_level(csv_dir, level)
        merged.to_csv(os.path.join(out_dir, f'merged_{level}_gitter.csv'), index=False)

save_merged_levels(r"C:\Users\petre\PycharmProjects\cleancensus\raw", r"C:\Users\petre\PycharmProjects\cleancensus\merged")

In [ ]:
# Add parent grid IDs (derive 1km/10km IDs from 100m/1km IDs)
import re
import pandas as pd

_ID_RE = re.compile(r'^CRS3035RES(\d+)mN(\d+)E(\d+)$')


def parent_gitter_id(gid: str, target_res: int) -> str | None:
    """
    Convert any CRS3035RESXmN..E.. ID to the parent grid at target_res (in meters).
    Floors N/E to the target resolution and replaces the RES part.
    Examples:
      CRS3035RES1000mN2689000E4337000 -> target_res=10000 -> CRS3035RES10000mN2680000E4330000
      CRS3035RES100mN2689400E4337600  -> target_res=1000  -> CRS3035RES1000mN2689000E4337000
    """
    if not isinstance(gid, str):
        return None
    m = _ID_RE.match(gid.strip())
    if not m:
        return None
    _, n_str, e_str = m.groups()
    n = int(n_str)
    e = int(e_str)
    # floor to target resolution
    n_parent = (n // target_res) * target_res
    e_parent = (e // target_res) * target_res
    return f'CRS3035RES{target_res}mN{n_parent}E{e_parent}'


def add_parent_ids_for_level(df: pd.DataFrame, level: str) -> pd.DataFrame:
    """
    level in {"100m","1km"}.
    - If "1km": creates GITTER_ID_10km from GITTER_ID_1km.
    - If "100m": creates GITTER_ID_1km and GITTER_ID_10km from GITTER_ID_100m.
    Returns the same DataFrame (mutated) for convenience.
    """
    assert level in {"100m", "1km"}
    if level == "1km":
        id_col = "GITTER_ID_1km"
        df["GITTER_ID_10km"] = df[id_col].apply(lambda x: parent_gitter_id(x, 10000))
    else:  # "100m"
        id_col = "GITTER_ID_100m"
        df["GITTER_ID_1km"] = df[id_col].apply(lambda x: parent_gitter_id(x, 1000))
        df["GITTER_ID_10km"] = df[id_col].apply(lambda x: parent_gitter_id(x, 10000))
    return df

df_1km = pd.read_csv(r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter.csv")
df_1km = add_parent_ids_for_level(df_1km, "1km")
df_1km.to_csv(r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter.csv", index=False)

df_100m = pd.read_csv(r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter.csv")
df_100m = add_parent_ids_for_level(df_100m, "100m")
df_100m.to_csv(r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter.csv", index=False)

In [ ]:
# Sanity check for merged vs raw products
import os
import re
import glob
import pandas as pd
from collections import defaultdict

MERGED_DIR = r"C:\Users\petre\PycharmProjects\cleancensus\merged"
RAW_DIR = r"C:\Users\petre\PycharmProjects\cleancensus\raw"

ID_BY_LEVEL = {"100m": "GITTER_ID_100m", "1km": "GITTER_ID_1km", "10km": "GITTER_ID_10km"}
XY_BY_LEVEL = {"100m": ("x_mp_100m", "y_mp_100m"),
               "1km": ("x_mp_1km", "y_mp_1km"),
               "10km": ("x_mp_10km", "y_mp_10km")}


def read_csv_robust(path: str) -> pd.DataFrame:
    try:
        return pd.read_csv(path, sep=';', dtype=str, encoding='utf-8', encoding_errors='replace')
    except UnicodeDecodeError:
        return pd.read_csv(path, sep=';', dtype=str, encoding='ISO-8859-1', encoding_errors='replace')


def read_csv_auto(path: str) -> pd.DataFrame:
    # merged are usually comma-separated; fall back to semicolon
    try:
        return pd.read_csv(path, dtype=str, encoding_errors='replace')
    except Exception:
        return pd.read_csv(path, sep=';', dtype=str, encoding_errors='replace')


_ID_RE = re.compile(r'^CRS3035RES(\d+)mN(\d+)E(\+?\-?\d+)$')


def parent_gitter_id(gid: str, target_res: int) -> str | None:
    if not isinstance(gid, str): return None
    m = _ID_RE.match(gid.strip())
    if not m: return None
    _, n_str, e_str = m.groups()
    n = int(n_str);
    e = int(e_str)
    n_parent = (n // target_res) * target_res
    e_parent = (e // target_res) * target_res
    return f'CRS3035RES{target_res}mN{n_parent}E{e_parent}'


def pick_total_columns(df: pd.DataFrame) -> list[str]:
    # Heuristic: columns that look like totals
    patterns = [
        r'^Einwohner_', r'^Insgesamt_', r'_Insgesamt$', r'Bevoelkerungszahl', r'Haushalte', r'Wohnungen'
    ]
    cols = []
    for c in df.columns:
        if c in ID_BY_LEVEL.values(): continue
        if any(re.search(p, c) for p in patterns):
            # numeric-ish?
            try:
                pd.to_numeric(df[c], errors='coerce')
                cols.append(c)
            except Exception:
                pass
    # Keep first few to avoid noise
    return cols[:5]


def numeric_summary(series: pd.Series) -> dict:
    s = pd.to_numeric(series, errors='coerce')
    return {
        "non_null": int(s.notna().sum()),
        "negatives": int((s < 0).sum()),
        "sum": float(s.sum(skipna=True)) if s.notna().any() else 0.0
    }


def report_header(title: str):
    print("=" * len(title));
    print(title);
    print("=" * len(title))


# 1) Load merged files
merged_files = {
    "100m": os.path.join(MERGED_DIR, "merged_100m_gitter.csv"),
    "1km": os.path.join(MERGED_DIR, "merged_1km_gitter.csv"),
    "10km": os.path.join(MERGED_DIR, "merged_10km_gitter.csv"),
}
present_levels = [lvl for lvl, p in merged_files.items() if os.path.exists(p)]
report_header(f"Merged files detected: {present_levels}")

merged = {}
for lvl in present_levels:
    df = read_csv_auto(merged_files[lvl])
    # basic cleanup
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].str.strip()
    merged[lvl] = df

# 2) Structural checks per merged file
for lvl, df in merged.items():
    report_header(f"Structure checks | {lvl}")
    id_col = ID_BY_LEVEL[lvl]
    x_col, y_col = XY_BY_LEVEL[lvl]

    print(f"- Rows: {len(df):,}, Cols: {len(df.columns):,}")
    print(f"- Has ID column '{id_col}': {id_col in df.columns}")
    if id_col in df.columns:
        null_ids = int(df[id_col].isna().sum())
        dup_ids = int((len(df[id_col]) - df[id_col].nunique()))
        print(f"  Null IDs: {null_ids:,}, Duplicate IDs: {dup_ids:,}")

    # x/y existence: exactly once each (if present at all)
    x_count = sum(c == x_col for c in df.columns)
    y_count = sum(c == y_col for c in df.columns)
    print(f"- {x_col} count: {x_count}, {y_col} count: {y_count}")

    # Total-like columns quick stats
    totals = pick_total_columns(df)
    if totals:
        print(f"- Sample total-like columns ({len(totals)}): {totals}")
        for c in totals:
            stats = numeric_summary(df[c])
            print(f"  {c} -> non_null={stats['non_null']:,}, negatives={stats['negatives']:,}, sum={stats['sum']:.2f}")
    else:
        print("- No obvious total-like columns found (heuristic).")

    # NaN ratio (overall)
    nan_ratio = float(df.isna().sum().sum()) / (len(df) * max(1, len(df.columns)))
    print(f"- Overall NaN ratio: {nan_ratio:.3%}")


# 3) Hierarchy consistency (if multiple levels are present)
def check_hierarchy(child_lvl: str, parent_lvl: str, child_res: int, parent_res: int):
    if child_lvl not in merged or parent_lvl not in merged: return
    report_header(f"Hierarchy check {child_lvl} -> {parent_lvl}")
    cdf = merged[child_lvl].copy()
    pdf = merged[parent_lvl].copy()
    c_id = ID_BY_LEVEL[child_lvl]
    p_id = ID_BY_LEVEL[parent_lvl]

    # derive parent IDs in child
    derived_col = f"derived_{p_id}"
    cdf[derived_col] = cdf[c_id].apply(lambda x: parent_gitter_id(x, parent_res))

    # coverage: parent IDs from child vs parent file
    child_parents = set(cdf[derived_col].dropna().unique())
    true_parents = set(pdf[p_id].dropna().unique())
    coverage = len(child_parents & true_parents) / max(1, len(child_parents))
    print(f"- Parent ID coverage (child-derived in parent file): {coverage:.2%}  "
          f"({len(child_parents & true_parents):,}/{len(child_parents):,})")

    # If we can find a total column in both, compare sums aggregated to parent
    c_totals = pick_total_columns(cdf)
    p_totals = pick_total_columns(pdf)
    anchor_child = c_totals[0] if c_totals else None
    anchor_parent = p_totals[0] if p_totals else None
    if anchor_child and anchor_parent:
        csum = (pd.to_numeric(cdf[anchor_child], errors='coerce')
                .groupby(cdf[derived_col]).sum(min_count=1))
        psum = pd.to_numeric(pdf.set_index(p_id)[anchor_parent], errors='coerce')
        joined = pd.DataFrame({"child_sum": csum}).join(psum.rename("parent_sum"), how="inner")
        # relative difference on overlapping parents
        joined["reldiff"] = (joined["child_sum"] - joined["parent_sum"]) / joined["parent_sum"].replace(0, pd.NA)
        print(f"- Anchor totals compared (first matching columns):")
        print(f"  child col: {anchor_child}")
        print(f"  parent col: {anchor_parent}")
        print(f"  Overlap cells: {len(joined):,}")
        print(f"  |reldiff| median={joined['reldiff'].abs().median(skipna=True):.4f}, "
              f"p95={joined['reldiff'].abs().quantile(0.95):.4f}")
    else:
        print("- Skipped totals comparison (no suitable anchor columns found).")


# Check 100m->1km and 1km->10km
check_hierarchy("100m", "1km", child_res=100, parent_res=1000)
check_hierarchy("1km", "10km", child_res=1000, parent_res=10000)

# 4) Compare merged ID coverage to RAW (some originals)
for lvl in present_levels:
    report_header(f"Compare merged {lvl} IDs with RAW coverage")
    id_col = ID_BY_LEVEL[lvl]
    # find some raw files for the level
    raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, f"*{lvl}-Gitter*.csv")))
    if not raw_paths:
        print("- No RAW files found for this level.")
        continue

    merged_ids = set(merged[lvl][id_col].dropna().unique())
    # Collect union of IDs from a few RAW files (first 3 to keep it quick)
    raw_ids_union = set()
    for rp in raw_paths[:3]:
        rdf = read_csv_robust(rp)
        if id_col in rdf.columns:
            raw_ids_union |= set(rdf[id_col].dropna().unique())

    if raw_ids_union:
        cov = len(merged_ids & raw_ids_union) / max(1, len(raw_ids_union))
        print(f"- Coverage of RAW IDs in merged: {cov:.2%} "
              f"({len(merged_ids & raw_ids_union):,}/{len(raw_ids_union):,})  [RAW files sampled: {min(3, len(raw_paths))}]")
        # Check if merged has any IDs not present in raw sample
        extra = len(merged_ids - raw_ids_union)
        print(f"- Extra IDs present only in merged (vs. sampled RAW): {extra:,}")
    else:
        print("- RAW files sampled, but no ID column present or empty.")

print("\nSanity checks complete.")

In [ ]:
# Collapse duplicate population totals and adjust them proportionally to the parent level (floats)
import re
import pandas as pd

# Reuse the same parent ID derivation logic
_ID_RE = re.compile(r'^CRS3035RES(\d+)mN(\d+)E(\+?\-?\d+)$')


def parent_gitter_id(gid: str, target_res: int) -> str | None:
    if not isinstance(gid, str): return None
    m = _ID_RE.match(gid.strip())
    if not m: return None
    _, n_str, e_str = m.groups()
    n = int(n_str);
    e = int(e_str)
    n_parent = (n // target_res) * target_res
    e_parent = (e // target_res) * target_res
    return f'CRS3035RES{target_res}mN{n_parent}E{e_parent}'


def detect_total_columns(df: pd.DataFrame, level: str) -> list[str]:
    # Identify candidate population-total columns for a level
    patt = re.compile(rf'^(Insgesamt_Bevoelkerung_.*_{level}-Gitter|Einwohner_Bevoelkerungszahl_{level}-Gitter)$')
    return [c for c in df.columns if patt.match(c)]


def collapse_population_totals(df: pd.DataFrame, level: str, tol: float = 1e-6,
                               verbose: bool = True, sample_n: int = 20):
    """
    Collapse multiple population total columns at a given level into a single consensus column:
    - For each row, pick the value supported by the majority of total columns (within tolerance).
    - On ties, pick the group closest to the row median; if still tied, pick the first encountered.
    - The consensus value is the median of the winning group (robust).
    - Returns (df, pop_col_name) where pop_col_name = f"POP_TOTAL_{level}".

    Detailed reporting is printed when verbose=True.
    """
    # Identify candidate population total columns for the level
    pop_cols = [c for c in df.columns if re.match(r'^(Einwohner_Bevoelkerungszahl|Insgesamt_Bevoelkerung_)', c)]
    if not pop_cols:
        raise ValueError(f"No population total columns found for level='{level}'")

    # Ensure numeric columns
    for c in pop_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    id_col = f"GITTER_ID_{level}" if f"GITTER_ID_{level}" in df.columns else None
    values = df[pop_cols].copy()

    # Helper to compute a consensus for one row
    def row_consensus(row_vals: pd.Series):
        # Collect non-null values
        non_null = row_vals.dropna()
        total_non_na = len(non_null)
        if total_non_na == 0:
            return np.nan, 0, 0, False, {}  # no data

        # Build tolerance-aware groups: each group is represented by a list of indices and values
        # Strategy: iterate values; place into an existing group if within tol of that group's representative,
        # otherwise start a new group. Group representative is the running median for robustness.
        groups = []  # list of dicts: {idxs: [...], vals: [...], rep: float}
        for col_name, val in non_null.items():
            placed = False
            for g in groups:
                if abs(val - g["rep"]) <= tol:
                    g["idxs"].append(col_name)
                    g["vals"].append(val)
                    g["rep"] = float(np.median(g["vals"]))
                    placed = True
                    break
            if not placed:
                groups.append({"idxs": [col_name], "vals": [val], "rep": float(val)})

        # Choose group with max size
        sizes = [len(g["idxs"]) for g in groups]
        max_size = max(sizes)
        winners = [g for g in groups if len(g["idxs"]) == max_size]
        tie = len(winners) > 1

        if tie:
            # Break tie by closeness of group rep to row median of all non-null values
            row_median = float(np.median(non_null.values))
            winners.sort(key=lambda g: abs(g["rep"] - row_median))
        winner = winners[0]
        consensus_value = float(np.median(winner["vals"]))  # robust consensus

        # Build support map for reporting
        support_map = {g["rep"]: g["idxs"] for g in groups}
        return consensus_value, len(winner["idxs"]), total_non_na, tie, support_map

    # Compute consensus for all rows
    cons_vals = []
    cons_support = []
    cons_total_non_na = []
    cons_tie = []
    cons_support_maps = []

    # Apply row-wise; number of pop_cols is usually small, so this is fine
    for i in range(len(values)):
        row = values.iloc[i]
        v, sup, tot, tie, s_map = row_consensus(row)
        cons_vals.append(v)
        cons_support.append(sup)
        cons_total_non_na.append(tot)
        cons_tie.append(tie)
        cons_support_maps.append(s_map)

    pop_col = f"POP_TOTAL_{level}"
    df[pop_col] = cons_vals

    # Reporting and diagnostics
    if verbose:
        cons_support = pd.Series(cons_support, index=df.index)
        cons_total_non_na = pd.Series(cons_total_non_na, index=df.index)
        cons_tie = pd.Series(cons_tie, index=df.index)

        unanimity = (cons_support == cons_total_non_na) & (cons_total_non_na > 0)
        unanimous_rows = int(unanimity.sum())
        conflicted_rows = int((~unanimity & (cons_total_non_na > 0)).sum())
        tie_rows = int(cons_tie.sum())

        print("=" * 72)
        print(f"Collapsed population totals -> {pop_col}")
        print(f"- Rows: {len(df):,}")
        print(f"- Unanimous rows: {unanimous_rows:,}")
        print(f"- Rows with conflicts resolved by majority: {conflicted_rows:,}")
        print(f"- Rows with tie among groups (resolved by median proximity): {tie_rows:,}")

        # Per-column disagreement counts vs consensus
        per_col_disagree = {}
        per_col_absdiff_stats = {}
        for c in pop_cols:
            col_vals = df[c]
            # Compare to consensus with tolerance, treating both NaN as ok
            both_na = col_vals.isna() & pd.Series(df[pop_col]).isna()
            abs_diff = (col_vals.astype(float) - df[pop_col].astype(float)).abs()
            disagree_mask = (~both_na) & (abs_diff > tol)
            per_col_disagree[c] = int(disagree_mask.sum())

            # Stats of abs diff for rows where both are non-null
            comparable = (~col_vals.isna()) & (~df[pop_col].isna())
            if comparable.any():
                diffs = abs_diff[comparable]
                q = diffs.quantile([0.0, 0.5, 0.95, 1.0])
                per_col_absdiff_stats[c] = {
                    "min": float(q.iloc[0]),
                    "median": float(q.iloc[1]),
                    "p95": float(q.iloc[2]),
                    "max": float(q.iloc[3]),
                }
            else:
                per_col_absdiff_stats[c] = {"min": 0.0, "median": 0.0, "p95": 0.0, "max": 0.0}

        print("- Per-column disagreements vs consensus (count of rows beyond tol):")
        for c in pop_cols:
            print(f"  {c}: {per_col_disagree[c]:,}")

        print("- Per-column abs diff stats vs consensus (where comparable):")
        for c in pop_cols:
            s = per_col_absdiff_stats[c]
            print(f"  {c}: min={s['min']:.6g}, median={s['median']:.6g}, p95={s['p95']:.6g}, max={s['max']:.6g}")

        # Show sample of conflicting rows with details
        if conflicted_rows > 0:
            # Identify conflicting rows
            conflict_mask = ~unanimity & (cons_total_non_na > 0)
            conflicts_idx = df.index[conflict_mask]
            # Build a small table: ID (if any), all pop_cols, consensus, support/total, tie flag
            sample_idx = conflicts_idx[:sample_n]

            view_cols = ([id_col] if id_col else []) + pop_cols + [pop_col]
            sample_df = df.loc[sample_idx, view_cols].copy()
            sample_df["cons_support"] = cons_support.loc[sample_idx].astype(int)
            sample_df["cons_total_cols"] = cons_total_non_na.loc[sample_idx].astype(int)
            sample_df["tie"] = cons_tie.loc[sample_idx].astype(bool)

            print(f"- Sample of {len(sample_idx)} conflicting rows (showing all totals and consensus):")
            print(sample_df.to_string(index=False, justify='left', max_colwidth=80))

        print("=" * 72)

    return df, pop_col


def ensure_parent_column(df: pd.DataFrame, level: str) -> tuple[pd.DataFrame, str]:
    """
    Return df with a parent ID column added if missing, and the parent column name.
    100m -> adds/uses GITTER_ID_1km
    1km  -> adds/uses GITTER_ID_10km
    10km -> no parent
    """
    if level == "100m":
        child_id, parent_id, tgt = "GITTER_ID_100m", "GITTER_ID_1km", 1000
    elif level == "1km":
        child_id, parent_id, tgt = "GITTER_ID_1km", "GITTER_ID_10km", 10000
    else:
        raise ValueError("Only 100m and 1km have parents for this adjustment.")
    if parent_id not in df.columns:
        df[parent_id] = df[child_id].apply(lambda x: parent_gitter_id(x, tgt))
    return df, parent_id


def proportional_adjust_to_parent(child_df: pd.DataFrame, parent_df: pd.DataFrame,
                                  child_level: str, parent_level: str,
                                  child_pop_col: str, parent_pop_col: str) -> pd.DataFrame:
    """
    Scales child POP_TOTAL so that group sums match the parent POP_TOTAL (floats).
    Factor per parent = parent_total / sum(child_total_in_group). Where group sum is 0, uses factor 1.
    """
    child_df, parent_col = ensure_parent_column(child_df, child_level)

    # Parent totals by parent ID
    parent_id_col = f"GITTER_ID_{parent_level}"
    if parent_id_col not in parent_df.columns:
        raise ValueError(f"{parent_id_col} missing in parent_df.")
    p_tot = (parent_df[[parent_id_col, parent_pop_col]]
             .dropna(subset=[parent_id_col])
             .copy())
    p_tot[parent_pop_col] = pd.to_numeric(p_tot[parent_pop_col], errors='coerce')
    p_tot = p_tot.groupby(parent_id_col)[parent_pop_col].sum(min_count=1)

    # Child sums by parent
    child_df[child_pop_col] = pd.to_numeric(child_df[child_pop_col], errors='coerce')
    c_sum = child_df.groupby(parent_col)[child_pop_col].sum(min_count=1)

    # Build factor table
    factors = (p_tot / c_sum).rename("scale")
    factors = factors.replace([pd.NA, pd.NaT], 1.0).fillna(1.0)
    # Attach and scale
    child_df = child_df.merge(factors.to_frame(), left_on=parent_col, right_index=True, how="left")
    child_df["scale"] = child_df["scale"].replace([pd.NA, pd.NaT], 1.0).fillna(1.0)
    child_df[child_pop_col] = child_df[child_pop_col].astype(float) * child_df["scale"].astype(float)
    # child_df = child_df.drop(columns=["scale"])
    # Print diagnostics
    print("=" * 72)
    print(f"Proportional adjustment to parent level {parent_level} for {child_level} level")
    print(f"Maximum scale factor: {factors.max():.3f}")
    print(f"Average scale factor: {factors.mean():.3f}")
    return child_df


def adjust_population_pipeline(child_csv: str, parent_csv: str,
                               child_level: str, parent_level: str,
                               out_csv: str,
                               tol: float = 1e-6):
    """
    1) Load child and parent CSVs.
    2) Collapse population total columns to POP_TOTAL_<level> (verifies equality).
    3) Adjust child totals proportionally to parent totals (floats).
    4) Save result.
    """
    child_df = pd.read_csv(child_csv, dtype=str, encoding_errors='replace')
    parent_df = pd.read_csv(parent_csv, dtype=str, encoding_errors='replace')

    child_df, child_pop_col = collapse_population_totals(child_df, child_level, tol=tol)
    parent_df, parent_pop_col = collapse_population_totals(parent_df, parent_level, tol=tol)

    child_df = proportional_adjust_to_parent(child_df, parent_df,
                                             child_level, parent_level,
                                             child_pop_col, parent_pop_col)
    child_df.to_csv(out_csv, index=False)
    print(f"Saved proportionally adjusted {child_level} to {out_csv}")


def collapse_population_file(input_csv: str, level: str, output_csv: str, tol: float = 1e-6):
    df = pd.read_csv(input_csv, dtype=str, encoding_errors='replace')
    df, pop_col = collapse_population_totals(df, level, tol=tol)
    df.to_csv(output_csv, index=False)
    print(f"Collapsed totals for {level} -> {pop_col}; saved to {output_csv}")


# # 1) Collapse 10km totals (no parent adjustment)
ten_in  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_10km_gitter.csv"
ten_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_10km_gitter_pop_totals.csv"
collapse_population_file(ten_in, "10km", ten_out)

# 2) Collapse 1km totals and adjust to 10km (floats)
one_in  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter.csv"
one_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter_pop_totals.csv"
adjust_population_pipeline(
    child_csv=one_in,
    parent_csv=ten_out,
    child_level="1km", parent_level="10km",
    out_csv=one_out,
    tol=1e-6
)

# 3) Collapse 100m totals and adjust to 1km (floats)
hun_in  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter.csv"
hun_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter_pop_totals.csv"
adjust_population_pipeline(
    child_csv=hun_in,
    parent_csv=one_out,
    child_level="100m", parent_level="1km",
    out_csv=hun_out,
    tol=1e-6
)


In [ ]:
# Sanity checks
import re
import pandas as pd
import numpy as np

ten_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_10km_gitter_pop_totals.csv"
one_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter_pop_totals.csv"
hun_out = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter_pop_totals.csv"

_ID_RE = re.compile(r'^CRS3035RES(\d+)mN(\d+)E(\+?\-?\d+)$')

def parent_gitter_id(gid: str, target_res: int) -> str | None:
    if not isinstance(gid, str): return None
    m = _ID_RE.match(gid.strip())
    if not m: return None
    _, n_str, e_str = m.groups()
    n = int(n_str); e = int(e_str)
    n_parent = (n // target_res) * target_res
    e_parent = (e // target_res) * target_res
    return f'CRS3035RES{target_res}mN{n_parent}E{e_parent}'

def _num(x):
    return pd.to_numeric(x, errors="coerce")

def _print_header(title):
    print("=" * 72)
    print(title)
    print("=" * 72)

def check_level(df: pd.DataFrame, level: str, id_col: str, pop_col: str, tol: float = 1e-6):
    issues = []

    # Presence
    if id_col not in df.columns:
        issues.append(f"Missing {id_col}")
    if pop_col not in df.columns:
        issues.append(f"Missing {pop_col}")
        return {"issues": issues}

    # Types
    df[pop_col] = _num(df[pop_col])

    # Uniqueness of IDs
    dup_ids = df[id_col].duplicated(keep=False).sum()
    if dup_ids:
        issues.append(f"Duplicate IDs at {level}: {dup_ids}")

    # Missing / negative totals
    n_na = df[pop_col].isna().sum()
    n_neg = (df[pop_col] < 0).sum()
    # Basic stats
    stats = df[pop_col].describe(percentiles=[0.5, 0.95]).to_dict()

    print(f"[{level}] rows={len(df):,}, {id_col} unique={df[id_col].nunique():,}")
    print(f"[{level}] NaN totals: {n_na:,}, Negative totals: {n_neg:,}")
    print(f"[{level}] Totals stats (non-null): count={int(stats.get('count',0)):,}, "
          f"min={stats.get('min',np.nan):.6g}, median={stats.get('50%',np.nan):.6g}, "
          f"p95={stats.get('95%',np.nan):.6g}, max={stats.get('max',np.nan):.6g}")

    return {"issues": issues, "n_na": int(n_na), "n_neg": int(n_neg), "stats": stats}

def check_parent_child(child_df: pd.DataFrame, parent_df: pd.DataFrame,
                       child_level: str, parent_level: str,
                       child_id_col: str, parent_id_col: str,
                       child_pop_col: str, parent_pop_col: str,
                       tol: float = 1e-6):
    """
    Verifies that sum(child_pop by parent) ~= parent_pop at the parent level.
    Returns discrepancy stats and counts.
    """
    # Ensure parent key on child
    if parent_id_col not in child_df.columns:
        # derive from GITTER_ID_100m -> GITTER_ID_1km (target 1000) or 1km -> 10km (target 10000)
        target = 1000 if child_level == "100m" else 10000
        child_df = child_df.copy()
        child_df[parent_id_col] = child_df[child_id_col].apply(lambda x: parent_gitter_id(x, target))

    # Numeric
    child_df[child_pop_col] = _num(child_df[child_pop_col])
    parent_df[parent_pop_col] = _num(parent_df[parent_pop_col])

    # Aggregate child to parent
    child_agg = (child_df
                 .dropna(subset=[parent_id_col])
                 .groupby(parent_id_col, dropna=False)[child_pop_col]
                 .sum(min_count=1)
                 .rename("child_sum"))
    parent_tot = (parent_df
                  .dropna(subset=[parent_id_col])
                  .set_index(parent_id_col)[parent_pop_col]
                  .rename("parent_sum"))

    comp = pd.concat([child_agg, parent_tot], axis=1)
    comparable = comp.dropna(subset=["child_sum", "parent_sum"]).copy()
    comparable["abs_diff"] = (comparable["child_sum"] - comparable["parent_sum"]).abs()
    comparable["rel_err"] = comparable["abs_diff"] / comparable["parent_sum"].replace(0, np.nan)

    # Metrics
    n_groups = len(comp)
    n_comp = len(comparable)
    over_tol = (comparable["abs_diff"] > tol).sum()
    q_abs = comparable["abs_diff"].quantile([0.5, 0.95, 1.0]) if n_comp else pd.Series(dtype=float)
    q_rel = comparable["rel_err"].quantile([0.5, 0.95, 1.0]) if n_comp else pd.Series(dtype=float)

    print(f"[{child_level} -> {parent_level}] parent groups: {n_groups:,}, comparable: {n_comp:,}")
    print(f"[{child_level} -> {parent_level}] abs diff: median={q_abs.get(0.5, np.nan):.6g}, "
          f"p95={q_abs.get(0.95, np.nan):.6g}, max={q_abs.get(1.0, np.nan):.6g}, "
          f"over_tol(>{tol}): {int(over_tol):,}")
    print(f"[{child_level} -> {parent_level}] rel err:  median={q_rel.get(0.5, np.nan):.6g}, "
          f"p95={q_rel.get(0.95, np.nan):.6g}, max={q_rel.get(1.0, np.nan):.6g}")

    # Show a few worst offenders
    if n_comp:
        worst = comparable.sort_values("abs_diff", ascending=False).head(5)
        print(f"[{child_level} -> {parent_level}] worst 5 abs diffs:")
        print(worst[["child_sum", "parent_sum", "abs_diff", "rel_err"]].to_string())

    # Global totals check
    child_total = np.nansum(child_df[child_pop_col].values)
    parent_total = np.nansum(parent_df[parent_pop_col].values)
    glob_abs = abs(child_total - parent_total)
    print(f"[{child_level} -> {parent_level}] global sums: child={child_total:.6g}, "
          f"parent={parent_total:.6g}, abs_diff={glob_abs:.6g}")

    return {
        "n_groups": int(n_groups),
        "n_comparable": int(n_comp),
        "over_tol": int(over_tol),
        "abs_diff": {
            "median": float(q_abs.get(0.5, np.nan)) if n_comp else np.nan,
            "p95": float(q_abs.get(0.95, np.nan)) if n_comp else np.nan,
            "max": float(q_abs.get(1.0, np.nan)) if n_comp else np.nan,
        },
        "rel_err": {
            "median": float(q_rel.get(0.5, np.nan)) if n_comp else np.nan,
            "p95": float(q_rel.get(0.95, np.nan)) if n_comp else np.nan,
            "max": float(q_rel.get(1.0, np.nan)) if n_comp else np.nan,
        },
        "global": {"child_total": float(child_total), "parent_total": float(parent_total), "abs_diff": float(glob_abs)},
    }

def run_sanity_checks(ten_csv: str, one_csv: str, hun_csv: str, tol: float = 1e-6):
    _print_header("Loading results CSVs")
    d10 = pd.read_csv(ten_csv, dtype=str, encoding_errors="replace")
    d1  = pd.read_csv(one_csv, dtype=str, encoding_errors="replace")
    d0  = pd.read_csv(hun_csv, dtype=str, encoding_errors="replace")
    print("Loaded:",
          f"10km={len(d10):,} rows, 1km={len(d1):,} rows, 100m={len(d0):,} rows")

    _print_header("Per-level quick checks")
    r10 = check_level(d10, "10km", "GITTER_ID_10km", "POP_TOTAL_10km", tol)
    r1  = check_level(d1,  "1km",  "GITTER_ID_1km",  "POP_TOTAL_1km",  tol)
    r0  = check_level(d0,  "100m", "GITTER_ID_100m", "POP_TOTAL_100m", tol)

    _print_header("Parent-child consistency checks")
    c1 = check_parent_child(
        child_df=d1, parent_df=d10,
        child_level="1km", parent_level="10km",
        child_id_col="GITTER_ID_1km", parent_id_col="GITTER_ID_10km",
        child_pop_col="POP_TOTAL_1km", parent_pop_col="POP_TOTAL_10km",
        tol=tol
    )
    c0 = check_parent_child(
        child_df=d0, parent_df=d1,
        child_level="100m", parent_level="1km",
        child_id_col="GITTER_ID_100m", parent_id_col="GITTER_ID_1km",
        child_pop_col="POP_TOTAL_100m", parent_pop_col="POP_TOTAL_1km",
        tol=tol
    )

    _print_header("Summary")
    print("Issues 10km:", r10.get("issues", []))
    print("Issues 1km :", r1.get("issues", []))
    print("Issues 100m:", r0.get("issues", []))

    return {"per_level": {"10km": r10, "1km": r1, "100m": r0},
            "consistency": {"1km_to_10km": c1, "100m_to_1km": c0}}

# Run
summary = run_sanity_checks(ten_out, one_out, hun_out, tol=1e-6)

In [ ]:
# -*- coding: utf-8 -*-
# Create atomic age distribution per cell
# - All NaNs/±inf in the entire input are set to 0
# - Hard constraints: national per-age totals and per-cell totals

import pandas as pd
import numpy as np
import re
from typing import Tuple, Dict

# ----------------------------
# Column names and bin maps
# ----------------------------

POP_COL = "POP_TOTAL_10km"

# INFR (fine) – columns -> inclusive age ranges
INFR_BINS_10KM = {
    "Unter3_Alter_INFR_10km-Gitter":      (0, 2),
    "a3bis5_Alter_INFR_10km-Gitter":      (3, 5),
    "a6bis9_Alter_INFR_10km-Gitter":      (6, 9),
    "a10bis15_Alter_INFR_10km-Gitter":    (10, 15),
    "a16bis18_Alter_INFR_10km-Gitter":    (16, 18),
    "a19bis24_Alter_INFR_10km-Gitter":    (19, 24),
    "a25bis39_Alter_INFR_10km-Gitter":    (25, 39),
    "a40bis59_Alter_INFR_10km-Gitter":    (40, 59),
    "a60bis66_Alter_INFR_10km-Gitter":    (60, 66),
    "a67bis74_Alter_INFR_10km-Gitter":    (67, 74),
    "a75undaelter_Alter_INFR_10km-Gitter":(75, 100),  # top-coded
}

# 10-year groups
TENYEAR_BINS_10KM = {
    "Unter10_Alter_in_10er-Jahresgruppen_10km-Gitter": (0, 9),
    "a10bis19_Alter_in_10er-Jahresgruppen_10km-Gitter":(10, 19),
    "a20bis29_Alter_in_10er-Jahresgruppen_10km-Gitter":(20, 29),
    "a30bis39_Alter_in_10er-Jahresgruppen_10km-Gitter":(30, 39),
    "a40bis49_Alter_in_10er-Jahresgruppen_10km-Gitter":(40, 49),
    "a50bis59_Alter_in_10er-Jahresgruppen_10km-Gitter":(50, 59),
    "a60bis69_Alter_in_10er-Jahresgruppen_10km-Gitter":(60, 69),
    "a70bis79_Alter_in_10er-Jahresgruppen_10km-Gitter":(70, 79),
    "a80undaelter_Alter_in_10er-Jahresgruppen_10km-Gitter": (80, 100),
}

# 5 classes
FIVECLASS_BINS_10KM = {
    "Unter18_Alter_in_5_Altersklassen_10km-Gitter": (0, 17),
    "a18bis29_Alter_in_5_Altersklassen_10km-Gitter": (18, 29),
    "a30bis49_Alter_in_5_Altersklassen_10km-Gitter": (30, 49),
    "a50bis64_Alter_in_5_Altersklassen_10km-Gitter": (50, 64),
    "a65undaelter_Alter_in_5_Altersklassen_10km-Gitter": (65, 100),
}

# ------------------------------------
# Helpers to build selector indices
# ------------------------------------

def _age_index_slice(lo, hi, top_age):
    hi = min(hi, top_age)
    return np.arange(lo, hi + 1, dtype=int)

def _collect_bins(df, bin_map, top_age):
    bins = []
    for col, (lo, hi) in bin_map.items():
        if col in df.columns:
            bins.append((col, _age_index_slice(lo, hi, top_age)))
    return bins

def _final_rake_to_margins(X, row_targets, col_targets, *, tol=1e-10, max_iter=10):
    """
    Tiny IPF loop to meet both hard margins:
      - row_targets: shape (n,)
      - col_targets: shape (A,)
    """
    for _ in range(max_iter):
        # enforce columns
        col_sums = X.sum(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            cscale = np.divide(col_targets, np.maximum(col_sums, tol))
        X *= cscale  # broadcast over rows

        # enforce rows
        row_sums = X.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            rscale = np.divide(row_targets[:, None], np.maximum(row_sums, tol))
        X *= rscale  # broadcast over cols

        # print(f"Final rake iter {_}, cscale: {cscale}, rscale: {rscale}")

        # quick convergence check on both margins
        if (
            np.allclose(X.sum(axis=0), col_targets, rtol=1e-9, atol=1e-9)
            and np.allclose(X.sum(axis=1), row_targets, rtol=1e-9, atol=1e-9)
        ):
            break

# ---------------------------------------------------------
# Main reconciliation: single-year ages per 10km cell
# ---------------------------------------------------------

def fit_single_years_10km(
    df10: pd.DataFrame,
    national_by_age: pd.Series,
    *,
    top_age: int = 100,
    # damping (how hard to move per pass)
    alpha_5c: float = 0.9,
    alpha_10y: float = 0.85,
    alpha_infr: float = 0.8,
    # trust toward LOCAL (0..1). Smaller = trust NATIONAL more.
    trust_5c: float = 0.99,
    trust_10y: float = 0.99,
    trust_infr: float = 0.99,
    inner_passes: int = 30,
    outer_iters: int = 10,
    tol_rel: float = 1e-6,
    verbose: bool = True
) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Build per-year ages (0..top_age) for each 10km cell via multiplicative fitting with
    national-weighted bin targets to prevent zero-wipeout.
    Returns DataFrame shape=(n_cells, top_age+1); columns are 0..top_age (100 is "100+").
    """
    df = df10.copy()

    # --- National vector 0..top_age (100 = top bucket)
    nat = national_by_age.copy()
    if not isinstance(nat.index, pd.RangeIndex) or not nat.index.equals(pd.RangeIndex(0, top_age + 1)):
        raise AssertionError("national_by_age must have index 0..top_age (100=100+).")
    nat = nat.astype(float).values  # (A,)
    A = top_age + 1

    nat_sum = float(nat.sum())
    if not np.isfinite(nat_sum) or nat_sum <= 0:
        raise ValueError("National vector invalid (sum<=0).")
    nat_share = nat / nat_sum  # (A,)

    # --- Totals: positive & scaled to national grand total
    totals_raw = pd.to_numeric(df[POP_COL], errors="coerce").fillna(0.0).to_numpy(float)
    total_cells = float(totals_raw.sum())
    if total_cells <= 0:
        raise ValueError("Sum of cell totals <= 0 after NaNs->0.")
    scale_all = nat_sum / total_cells
    totals = totals_raw * scale_all  # hard cell totals we will enforce

    # --- Build bin lists
    five_bins = _collect_bins(df, FIVECLASS_BINS_10KM, top_age)
    ten_bins  = _collect_bins(df, TENYEAR_BINS_10KM,  top_age)
    infr_bins = _collect_bins(df, INFR_BINS_10KM,     top_age)

    # --- Initialization: national profile scaled to each cell total
    X = np.outer(totals, nat_share)  # (n, A)

    # Precompute national shares per bin and per-age shares within-bin
    def _bin_nat_shares(idx):
        w = nat[idx].sum()
        bin_share = w / nat_sum if w > 0 else 0.0                # scalar share of total
        if w > 0:
            within = nat[idx] / w                                 # (len(idx),) sum=1
        else:
            within = np.full(idx.size, 1.0 / max(idx.size, 1))    # fallback uniform
        return bin_share, within

    bin_nat = {tuple(idx): _bin_nat_shares(idx) for _, idx in (five_bins + ten_bins + infr_bins)}

    # --- Outer iterations
    for it in range(outer_iters):
        for _ in range(inner_passes):
            _apply_bin_scalings_mixed(
                df, X, five_bins, totals, nat_share, bin_nat,
                alpha=alpha_5c, trust_local=trust_5c, eps=tol_rel
            )
            _apply_bin_scalings_mixed(
                df, X, ten_bins, totals, nat_share, bin_nat,
                alpha=alpha_10y, trust_local=trust_10y, eps=tol_rel
            )
            _apply_bin_scalings_mixed(
                df, X, infr_bins, totals, nat_share, bin_nat,
                alpha=alpha_infr, trust_local=trust_infr, eps=tol_rel
            )

            # Hard re-project to cell totals
            row_sums = X.sum(axis=1, keepdims=True)
            np.divide(totals[:, None], np.maximum(row_sums, tol_rel), out=row_sums)
            X *= row_sums

        _final_rake_to_margins(X, totals, nat)  # totals = per-cell, nat = national per-age

        if verbose:
            S = X.sum(axis=0)  # (A,)
            err = np.mean(np.abs(S - nat) / np.maximum(nat, 1.0))
            print(f"[fit ages] iter {it+1}/{outer_iters} national rel.err ~ {err:.3e}")

    age_cols = list(range(0, top_age + 1))
    out = pd.DataFrame(X, index=df.index, columns=age_cols)
    return out, totals # return totals to add onto the original df10


def _apply_bin_scalings_mixed(
    df, X, bins, totals, nat_share, bin_nat_cache, *, alpha: float, trust_local: float, eps: float
):
    """Scale each bin toward a blended target:
       target = trust_local * local_bin + (1 - trust_local) * (nat_bin_share * cell_total)
       - Uses national within-bin age shares for seeding and stability
       - Prevents local zeros from annihilating ages
    """
    if not bins:
        return
    n = X.shape[0]

    for col, idx in bins:
        y = pd.to_numeric(df[col], errors="coerce").fillna(0.0).to_numpy(float)  # (n,)

        key = tuple(idx)
        if key in bin_nat_cache:
            bin_share, within = bin_nat_cache[key]
        else:
            # fallback (shouldn't happen)
            w = nat_share[idx].sum()
            bin_share = w
            within = np.full(idx.size, 1.0 / max(idx.size, 1))

        # blended target per cell
        nat_expected = bin_share * totals                  # (n,)
        target = trust_local * y + (1.0 - trust_local) * nat_expected

        s = X[:, idx].sum(axis=1)                          # current bin sums per cell
        # scaling factor (damped)
        f = np.ones_like(s)
        good = (s > eps) & np.isfinite(s)
        f[good] = (np.maximum(target[good], 0.0) / s[good]) ** alpha
        # clip extreme moves to keep stability
        np.clip(f, 1e-6, 1e6, out=f)

        # seed cells where s≈0 but target>0 using national within-bin shares
        seed = (s <= eps) & (target > 0)
        if seed.any():
            X[np.ix_(seed, idx)] = target[seed, None] * within[None, :]

        # apply scaling
        X[:, idx] *= f[:, None]


def _reseed_zero_rows(X, totals, nat_share, eps=1e-9): # not needed
    row_sums = X.sum(axis=1)
    dead = row_sums <= eps
    if np.any(dead):
        X[dead, :] = totals[dead, None] * nat_share[None, :]

# ----------------------------
# Loader for national table
# ----------------------------

def load_national_single_years(csv_path, label_col="Alter", value_col="Zahl", top_age=100):
    """
    Reads a CSV with labels like "0 Jahre", "1 Jahre", ..., "100 Jahre"
    and returns a Series indexed 0..top_age (100=100+ bucket).
    """
    df = pd.read_csv(csv_path,sep="\t")

    tmp = df[[label_col, value_col]].copy()
    tmp.columns = ["label", "value"]
    tmp = tmp.dropna(subset=["label", "value"])

    ages = (
        tmp["label"].astype(str)
        .str.extract(r"^\s*(\d+)\s+jahr(?:e)?\s*$", flags=re.IGNORECASE)[0]
        .astype(float)
    )
    tmp["age"] = ages.clip(lower=0, upper=top_age).astype("Int64")
    tmp = tmp.dropna(subset=["age"])
    tmp["age"] = tmp["age"].astype(int)
    tmp["value"] = pd.to_numeric(tmp["value"], errors="coerce").fillna(0).astype(float)

    nat = tmp.groupby("age")["value"].sum().reindex(range(0, top_age + 1), fill_value=0.0)
    if nat.sum() <= 0:
        raise ValueError("National total is zero—check CSV.")
    return nat

# ----------------------------
# Run / I/O
# ----------------------------

if __name__ == "__main__":

    PATH_DF10 = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_10km_gitter_pop_totals.csv"
    PATH_NAT  = r"C:\Users\petre\PycharmProjects\cleancensus\raw\national_age_totals.csv"
    OUT_CSV = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.csv"
    OUT_PICKLE = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.pickle"

    df10_raw = pd.read_csv(PATH_DF10)

    # Global hygiene: NaN/±inf -> 0 across the ENTIRE input (non-age cols included)
    df10 = df10_raw.replace([np.inf, -np.inf], np.nan).fillna(0)

    if "GITTER_ID_10km" in df10.columns:
        df10 = df10.set_index("GITTER_ID_10km")

    nat = load_national_single_years(PATH_NAT, top_age=100)

    ages_per_cell, adjusted_cell_totals = fit_single_years_10km( # adjusted cell totals to national age vector
        df10,
        nat,
    )

    # Sanity checks

    assert np.allclose(ages_per_cell.sum(axis=0).values, nat.values)  # national per-age

    # Compute per-cell totals
    calc_totals = ages_per_cell.sum(axis=1).values
    given_totals = df10[POP_COL].to_numpy(float)

    # Differences
    diff = calc_totals - given_totals
    abs_diff = np.abs(diff)

    # Quick summary
    print("Max abs diff:", abs_diff.max())
    print("Mean abs diff:", abs_diff.mean())

    # Show diffs
    debug_df = pd.DataFrame({
        "calc_total": calc_totals,
        "given_total": given_totals,
        "diff": diff
    })
    display(debug_df[debug_df["diff"].abs() > 1e-6].head(50))

    # Allow tolerance of 0.02% (2e-4 relative error) on per-cell totals
    calc_totals = ages_per_cell.sum(axis=1).values
    given_totals = df10[POP_COL].to_numpy(float)

    rel_err = np.abs(calc_totals - given_totals) / np.maximum(given_totals, 1.0)
    max_rel_err = rel_err.max()
    print(f"Max per-cell relative error: {max_rel_err:.3e}")

    assert max_rel_err <= 2e-4, (
    f"Per-cell totals differ by more than 0.02% (max rel.err={max_rel_err:.3e})"
    )
    # But adj cell totals must fit exactly
    rel_err = np.abs(given_totals - adjusted_cell_totals) / np.maximum(adjusted_cell_totals, 1.0)
    max_rel_err = rel_err.max()
    print(f"Max adj cell relative error: {max_rel_err:.3e}")
    assert np.allclose(adjusted_cell_totals, calc_totals)


    # Merge back
    df10_with_ages = df10.join(ages_per_cell)
    df10_with_ages = df10_with_ages.rename(columns={i: f"AGE_{i}" for i in ages_per_cell.columns})

    df10_with_ages[f"{POP_COL}_adj"] = adjusted_cell_totals

    df10_with_ages.to_pickle(OUT_PICKLE)
    df10_with_ages.to_csv(OUT_CSV)

    # Post-infos
    cell_diff = df10_with_ages.loc[:, "AGE_0":"AGE_100"].sum(axis=1) - df10_with_ages[POP_COL]
    print("Max per-cell deviation:", float(cell_diff.abs().max()))

    nat_reconstructed = df10_with_ages.loc[:, "AGE_0":"AGE_100"].sum(axis=0).values
    diff_nat = nat_reconstructed - nat.values
    print("National total diff (sum):", float(diff_nat.sum()))
    print("Max per-age abs diff:", float(np.abs(diff_nat).max()))

    # Spotlight check if column exists
    test_col = "a30bis39_Alter_in_10er-Jahresgruppen_10km-Gitter"
    if test_col in df10_with_ages.columns:
        approx = df10_with_ages.loc[:, "AGE_30":"AGE_39"].sum(axis=1)
        orig = pd.to_numeric(df10_with_ages[test_col], errors="coerce").fillna(0.0)
        rel_err = (approx - orig) / orig.replace(0, np.nan)
        print("30–39 median rel. error:", float(rel_err.abs().median()))
        print(rel_err)

    print("Grand total check:",
          float(df10_with_ages.loc[:, "AGE_0":"AGE_100"].to_numpy().sum()),
          "vs", float(nat.sum()))


In [11]:
# -*- coding: utf-8 -*-
"""
Hierarchical downscaling of atomar ages:
10km (AGE_0..AGE_100) -> 1km -> 100m
with trust-mixed local aggregated bins and hard constraints:
- child row totals are HARD (given POP_TOTAL_* columns)
- parent per-age totals are HARD (sum across children per age equals parent)
"""

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, List, Tuple, Iterable, Optional
from tqdm import tqdm

# ---------------------------------------------------------------------
# 0) Bin definitions
# ---------------------------------------------------------------------

@dataclass(frozen=True)
class BinSpec:
    """Maps column name -> inclusive age range [lo, hi]."""
    cols_to_ranges: Dict[str, Tuple[int, int]]


def make_infr_bins(level: str, top_age: int = 100) -> BinSpec:
    # level in {"10km","1km","100m"} to pick the suffix
    suf = {"10km":"_10km-Gitter","1km":"_1km-Gitter","100m":"_100m-Gitter"}[level]
    return BinSpec({
        f"Unter3_Alter_INFR_{suf[1:]}":       (0, 2),
        f"a3bis5_Alter_INFR_{suf[1:]}":       (3, 5),
        f"a6bis9_Alter_INFR_{suf[1:]}":       (6, 9),
        f"a10bis15_Alter_INFR_{suf[1:]}":     (10, 15),
        f"a16bis18_Alter_INFR_{suf[1:]}":     (16, 18),
        f"a19bis24_Alter_INFR_{suf[1:]}":     (19, 24),
        f"a25bis39_Alter_INFR_{suf[1:]}":     (25, 39),
        f"a40bis59_Alter_INFR_{suf[1:]}":     (40, 59),
        f"a60bis66_Alter_INFR_{suf[1:]}":     (60, 66),
        f"a67bis74_Alter_INFR_{suf[1:]}":     (67, 74),
        f"a75undaelter_Alter_INFR_{suf[1:]}": (75, top_age),
    })


def make_tenyear_bins(level: str, top_age: int = 100) -> BinSpec:
    suf = {"10km":"_10km-Gitter","1km":"_1km-Gitter","100m":"_100m-Gitter"}[level]
    return BinSpec({
        f"Unter10_Alter_in_10er-Jahresgruppen{suf}": (0, 9),
        f"a10bis19_Alter_in_10er-Jahresgruppen{suf}": (10, 19),
        f"a20bis29_Alter_in_10er-Jahresgruppen{suf}": (20, 29),
        f"a30bis39_Alter_in_10er-Jahresgruppen{suf}": (30, 39),
        f"a40bis49_Alter_in_10er-Jahresgruppen{suf}": (40, 49),
        f"a50bis59_Alter_in_10er-Jahresgruppen{suf}": (50, 59),
        f"a60bis69_Alter_in_10er-Jahresgruppen{suf}": (60, 69),
        f"a70bis79_Alter_in_10er-Jahresgruppen{suf}": (70, 79),
        f"a80undaelter_Alter_in_10er-Jahresgruppen{suf}": (80, top_age),
    })


def make_fiveclass_bins(level: str, top_age: int = 100) -> BinSpec:
    suf = {"10km":"_10km-Gitter","1km":"_1km-Gitter","100m":"_100m-Gitter"}[level]
    return BinSpec({
        f"Unter18_Alter_in_5_Altersklassen{suf}": (0, 17),
        f"a18bis29_Alter_in_5_Altersklassen{suf}": (18, 29),
        f"a30bis49_Alter_in_5_Altersklassen{suf}": (30, 49),
        f"a50bis64_Alter_in_5_Altersklassen{suf}": (50, 64),
        f"a65undaelter_Alter_in_5_Altersklassen{suf}": (65, top_age),
    })


# ---------------------------------------------------------------------
# 1) Utilities
# ---------------------------------------------------------------------

def age_cols(top_age: int = 100) -> List[str]:
    return [f"AGE_{a}" for a in range(0, top_age + 1)]


def _age_idx(lo: int, hi: int, top_age: int) -> np.ndarray:
    hi = min(hi, top_age)
    return np.arange(lo, hi + 1, dtype=np.intp)


def _collect_existing_bins(df: pd.DataFrame, spec: BinSpec, top_age: int) -> List[Tuple[str, np.ndarray]]:
    out: List[Tuple[str, np.ndarray]] = []
    for col, (lo, hi) in spec.cols_to_ranges.items():
        if col in df.columns:
            out.append((col, _age_idx(lo, hi, top_age)))
    return out


# ---------------------------------------------------------------------
# 2) Raking/IPF with hard margins
# ---------------------------------------------------------------------
def _final_rake_to_margins(X, row_targets, col_targets, *, tol=1e-10, max_iter=10):
    """
    Tiny IPF loop to meet both hard margins:
      - row_targets: shape (n,)
      - col_targets: shape (A,)
    """
    for _ in range(max_iter):
        # enforce columns
        col_sums = X.sum(axis=0)
        with np.errstate(divide="ignore", invalid="ignore"):
            cscale = np.divide(col_targets, np.maximum(col_sums, tol))
        X *= cscale  # broadcast over rows

        # enforce rows
        row_sums = X.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            rscale = np.divide(row_targets[:, None], np.maximum(row_sums, tol))
        X *= rscale  # broadcast over cols

        # print(f"Final rake iter {_}, cscale: {cscale}, rscale: {rscale}")

        # quick convergence check on both margins
        if (
            np.allclose(X.sum(axis=0), col_targets, rtol=1e-9, atol=tol)
            and np.allclose(X.sum(axis=1), row_targets, rtol=1e-9, atol=tol)
        ):
            break

# ---------------------------------------------------------------------
# 3) Trust-mixing bin scaling at child level
# ---------------------------------------------------------------------

def _apply_child_bin_scalings_mixed(
    df_child: pd.DataFrame,
    X: np.ndarray,                   # (n_child, A) working matrix (in-place)
    child_totals: np.ndarray,        # (n_child,)
    parent_share: np.ndarray,        # (A,) shares sum to 1
    bins: List[Tuple[str, np.ndarray]],
    *,
    alpha: float = 0.85,
    trust_threshold: float = 200.0,
    eps: float = 1e-9,
) -> None:
    """
    For every bin and every child row i:
        target_i = w_i * local_bin_i + (1 - w_i) * (parent_bin_share * child_total_i)
    where w_i = min(1, child_total_i / trust_threshold),
          parent_bin_share = sum(parent_share[idx])
    Then scale X[i, idx] multiplicatively toward target_i (damped by alpha).
    """
    if not bins:
        return

    n, A = X.shape
    assert child_totals.shape == (n,)

    parent_bin_share_cache: Dict[Tuple[int,...], float] = {}

    # weight per child (0..1)
    # w = np.minimum(1.0, child_totals / max(trust_threshold, eps))
    w_min = 0.4
    w_base = np.minimum(1.0, child_totals / max(trust_threshold, eps))
    w = w_min + (1.0 - w_min) * w_base

    for col, idx in bins:
        # parent bin share (scalar)
        key = tuple(idx.tolist())
        p_share = parent_bin_share_cache.get(key)
        if p_share is None:
            p_share = float(parent_share[idx].sum())
            parent_bin_share_cache[key] = p_share

        # blended targets per child
        local_bin = pd.to_numeric(df_child[col], errors="coerce").fillna(0.0).to_numpy(float)
        nat_expected = p_share * child_totals
        target = w * local_bin + (1.0 - w) * nat_expected  # shape (n,)

        # current bin sums per child
        s = X[:, idx].sum(axis=1)  # (n,)

        # scaling factor (damped)
        f = np.ones_like(s)
        good = (s > eps) & np.isfinite(s)
        f[good] = (np.maximum(target[good], 0.0) / s[good]) ** alpha
        np.clip(f, 1e-6, 1e6, out=f)  # stability

        # seed if needed where s≈0 and target>0 using parent within-bin shares
        need_seed = (s <= eps) & (target > 0)
        if need_seed.any():
            # within-bin shares from parent (proportional to parent_share)
            pbin = parent_share[idx]
            pw = pbin.sum()
            if pw > 0:
                within = (pbin / pw)  # length |idx|
            else:
                within = np.full(idx.size, 1.0 / max(idx.size, 1))
            X[np.ix_(need_seed, idx)] = target[need_seed, None] * within[None, :]

        # apply scaling
        X[:, idx] *= f[:, None]

def make_child_totals_adj(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    parent_adj_col: str,   # e.g. "POP_TOTAL_10km_adj"
    child_pop_col: str,    # e.g. "POP_TOTAL_1km" or "POP_TOTAL_100m"
    out_col: str = None
) -> pd.DataFrame:
    """
    For each parent id, multiply child totals by a scalar so
    sum(children)_adj == parent_adj. Writes a new column on child_df.
    If a group's child sum is 0 but parent_adj > 0, split equally.
    """
    out_col = out_col or f"{child_pop_col}_adj"

    # normalize IDs to strings to avoid 1 vs "1" mismatches
    pids_parent = parent_df[parent_id_col].astype(str)
    pids_child  = child_df[child_parent_id_col].astype(str)

    # parent map: pid -> adjusted total
    pmap = parent_df.assign(_pid=pids_parent).set_index("_pid")[parent_adj_col].astype(float)

    # child original totals
    child_tot = pd.to_numeric(child_df[child_pop_col], errors="coerce").fillna(0.0)

    # group sums at child
    gsum = pd.Series(child_tot.values, index=pids_child).groupby(level=0).sum().astype(float)

    # target (parent adj) per group
    tgt = pmap.reindex(gsum.index).fillna(0.0)

    # scale per group (safe divide)
    scale = pd.Series(0.0, index=gsum.index)
    nonzero = gsum > 0
    scale.loc[nonzero] = tgt.loc[nonzero] / gsum.loc[nonzero]

    # broadcast to rows
    s = pids_child.map(scale).fillna(0.0)
    adj = child_tot * s

    # handle degenerate groups (gsum==0 & tgt>0) -> equal split
    deg_groups = tgt.index[(~nonzero) & (tgt > 0)]
    if len(deg_groups) > 0:
        # number of children per (degenerate) group
        deg_counts = pids_child.value_counts().reindex(deg_groups).fillna(0).astype(int)
        # per-capita amount per group (still group-indexed)
        per_cap_group = (tgt.loc[deg_groups] / deg_counts.replace(0, np.nan)).fillna(0.0)
        # map to rows so index matches child_df
        per_cap_row = pids_child.map(per_cap_group).fillna(0.0)
        use_deg = pids_child.isin(deg_groups)  # row-indexed mask
        adj.loc[use_deg] = per_cap_row.loc[use_deg].values

    child_df[out_col] = adj.values
    return child_df


# ---------------------------------------------------------------------
# 4) Core downscaler (one hierarchy step)
# ---------------------------------------------------------------------

@dataclass
class DownscaleConfig:
    top_age: int = 100
    alpha_infr: float = 0.80
    alpha_10y: float = 0.85
    alpha_5c: float = 0.90
    trust_threshold: float = 200.0
    inner_passes: int = 20
    outer_iters: int = 5
    tol_rel_row: float = 2e-4  # 0.02% for row total validation
    rake_tol: float = 1e-10
    rake_max_iter: int = 50


def downscale_single_years(
    parent_df: pd.DataFrame,
    child_df: pd.DataFrame,
    *,
    parent_id_col: str,
    child_parent_id_col: str,
    child_pop_col: str,                 # hard totals at child level
    parent_age_cols: Optional[List[str]] = None,  # default AGE_0..AGE_100
    child_level_for_bins: str,          # "1km" or "100m" to choose names
    cfg: DownscaleConfig = DownscaleConfig(),
) -> pd.DataFrame:
    """
    For each parent_id group in child_df:
      - init child X = totals * parent per-age shares
      - trust-mixed bin scaling (INFR, 10y, 5c) per child (damped)
      - rake to (rows=child totals HARD, cols=parent per-age totals HARD)
    Returns a DataFrame aligned to child_df index with AGE_0..AGE_top columns.
    """
    top_age = cfg.top_age
    Acols = parent_age_cols or age_cols(top_age)
    A = len(Acols)
    assert A == (top_age + 1), "parent_age_cols must cover 0..top_age"

    # bin sets present at child level (only those that exist are used)
    infr_bins  = _collect_existing_bins(child_df, make_infr_bins(child_level_for_bins, top_age), top_age)
    teny_bins  = _collect_existing_bins(child_df, make_tenyear_bins(child_level_for_bins, top_age), top_age)
    fivec_bins = _collect_existing_bins(child_df, make_fiveclass_bins(child_level_for_bins, top_age), top_age)

    # Ensure required columns exist
    for c in [parent_id_col]:
        if c not in parent_df.columns:
            raise KeyError(f"Missing column in parent_df: {c}")
    for c in [child_parent_id_col, child_pop_col]:
        if c not in child_df.columns:
            raise KeyError(f"Missing column in child_df: {c}")
    for c in Acols:
        if c not in parent_df.columns:
            raise KeyError(f"Missing parent per-age column: {c}")

    # Prepare output frame
    out = pd.DataFrame(index=child_df.index, columns=Acols, dtype=float)

    # We assume child_df already has a column equal to parent_df[parent_id_col] for grouping
    # Group children by their parent id
    groups = child_df.groupby(child_parent_id_col, sort=False)

    # Map parent per-age vectors (as ndarray) by parent_id for speed
    parent_age_map: Dict[object, np.ndarray] = {}
    for pid, psub in parent_df[[parent_id_col] + Acols].groupby(parent_id_col):
        parent_age_map[pid] = psub[Acols].astype(float).to_numpy().sum(axis=0)  # if parent_id unique, this is just the row

    # Process each parent group
    for pid, child_idx in tqdm(groups.groups.items()):
        # Obtain parent per-age totals p (A,)
        p = parent_age_map.get(pid)
        if p is None:
            # No parent ages: fill zeros and continue (or raise)
            out.loc[child_idx, :] = 0.0
            continue

        # Children slice
        cdf = child_df.loc[child_idx]
        totals = pd.to_numeric(cdf[child_pop_col], errors="coerce").fillna(0.0).to_numpy(float)  # (n,)
        n = totals.size
        if n == 0:
            continue

        Psum = float(p.sum())
        if Psum <= 0:
            # Parent has no population -> children must be zero
            out.loc[child_idx, :] = 0.0
            continue

        # Parent shares
        parent_share = (p / Psum)  # (A,)

        # Initialize X = totals * parent shares
        X = np.outer(totals, parent_share)  # (n, A)

        # --- Outer iterations ---
        for _ in range(cfg.outer_iters):
            # Inner passes: trust-mixed bin scalings
            for _inner in range(cfg.inner_passes):
                if infr_bins:
                    _apply_child_bin_scalings_mixed(
                        cdf, X, totals, parent_share, infr_bins,
                        alpha=cfg.alpha_infr, trust_threshold=cfg.trust_threshold
                    )
                if teny_bins:
                    _apply_child_bin_scalings_mixed(
                        cdf, X, totals, parent_share, teny_bins,
                        alpha=cfg.alpha_10y, trust_threshold=cfg.trust_threshold
                    )
                if fivec_bins:
                    _apply_child_bin_scalings_mixed(
                        cdf, X, totals, parent_share, fivec_bins,
                        alpha=cfg.alpha_5c, trust_threshold=cfg.trust_threshold
                    )
                # re-project rows to hard child totals
                rs = X.sum(axis=1, keepdims=True)
                np.divide(totals[:, None], np.maximum(rs, 1e-12), out=rs)
                X *= rs

            # Across-children hard per-age = parent per-age
            _final_rake_to_margins(
                X, row_targets=totals, col_targets=p,
                tol=cfg.rake_tol, max_iter=cfg.rake_max_iter
            )

            # col_rel = np.abs(X.sum(axis=0) - p) / np.maximum(p, 1e-12)
            # print(f"Per-age rel.err: max={col_rel.max():.3e}, mean={col_rel.mean():.3e}")

        # Final rake to (rows=child totals HARD, cols=parent per-age totals HARD)
        _final_rake_to_margins(X, row_targets=totals, col_targets=p, tol=cfg.rake_tol, max_iter=150)


        # write result
        out.loc[child_idx, :] = X

        # print(f"Processed {pid} ({n} children)")

        # validations (per parent group)
        # 1) rows within 0.02%
        calc_totals = X.sum(axis=1)
        rel_err = np.abs(calc_totals - totals) / np.maximum(totals, 1.0)
        max_rel = float(rel_err.max(initial=0.0))
        if max_rel > cfg.tol_rel_row:
            raise AssertionError(
                f"[{child_level_for_bins}] Row totals dev > 0.02% for parent {pid}. "
                f"Max rel.err={max_rel:.3e}."
            )

        # 2) columns equal parent per-age (tight tol)
        child_totals = X.sum(axis=0)
        if not np.allclose(child_totals, p, rtol=0.0, atol=1e-5):
            diff = child_totals - p
            rel = np.divide(diff, np.maximum(np.abs(p), 1e-12))
            print("\n\n[DEBUG mismatch]")
            print(f"Parent ID: {pid}")
            print(f"Level: {child_level_for_bins}")
            print("Parent totals (p):")
            print(p)
            print("Child sums (X.sum(axis=0)):")
            print(child_totals)
            print("Difference (child - parent):")
            print(diff)
            print("Relative difference:")
            print(rel)
            if not np.allclose(child_totals, p, rtol=0.0, atol=1): # with very large cells atol may get closeish
                raise AssertionError(
                    f"[{child_level_for_bins}] Col totals != parent per-age for parent {pid}."
                )

    return out


# ---------------------------------------------------------------------
# 5) Run
# ---------------------------------------------------------------------

if __name__ == "__main__":

    PATH_DF10 = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.pickle"
    PATH_DF1  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_1km_gitter_pop_totals.csv"
    PATH_DF100 = r"C:\Users\petre\PycharmProjects\cleancensus\merged\merged_100m_gitter_pop_totals.csv"

    OUT_1KM   = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df1_with_single_years.pickle"
    OUT_1KM_CSV = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df1_with_single_years.csv"
    OUT_100M  = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df100_with_single_years.pickle"
    OUT_100M_CSV = r"C:\Users\petre\PycharmProjects\cleancensus\merged\df100_with_single_years.csv"

    df10 = pd.read_pickle(PATH_DF10).reset_index()  # has GITTER_ID_10km + AGE_* + POP_TOTAL_10km_adj
    df1  = pd.read_csv(PATH_DF1, dtype={"GITTER_ID_10km": str, "GITTER_ID_1km": str}, low_memory=False)
    df100 = pd.read_csv(PATH_DF100, dtype={"GITTER_ID_1km": str}, low_memory=False)

    df1 = df1.replace([np.inf, -np.inf], np.nan).fillna(0)
    df100 = df100.replace([np.inf, -np.inf], np.nan).fillna(0)

# Consistency checks
# -----------------------------------------------------------------------------------------------------------


    # --- 0) Normalize ID columns to clean comparisons --------------------------------
    def _norm_id(s: pd.Series) -> pd.Series:
        return s.astype(str).str.strip()

    def _topn(x, n=15):
        x = list(x)
        return x[:n] if len(x) > n else x

    def _report_link(parent_df, child_df, *, parent_key, child_parent_key,
                     parent_total_col=None, child_total_col=None, tag=""):
        pids = set(_norm_id(parent_df[parent_key]).unique())
        cids = set(_norm_id(child_df[child_parent_key]).unique())

        missing_in_child = pids - cids       # parent IDs that have NO children
        orphans_in_child = cids - pids       # child rows that reference UNKNOWN parents

        # Totals for impact (optional if total columns exist)
        parent_missing_total = None
        child_orphan_total  = None
        if parent_total_col and parent_total_col in parent_df:
            parent_missing_total = float(
                parent_df[parent_df[parent_key].astype(str).isin(missing_in_child)]
                [parent_total_col].fillna(0).astype(float).sum()
            )
        if child_total_col and child_total_col in child_df:
            child_orphan_total = float(
                child_df[child_df[child_parent_key].astype(str).isin(orphans_in_child)]
                [child_total_col].fillna(0).astype(float).sum()
            )

        print("\n" + "="*88)
        print(f"[COVERAGE] {tag}")
        print("-"*88)
        print(f"Unique parent IDs: {len(pids)}")
        print(f"Unique child parent-IDs: {len(cids)}")
        print(f"Parents with NO children: {len(missing_in_child)}"
              + (f" | parent-pop-sum≈{parent_missing_total:,.0f}" if parent_missing_total is not None else ""))
        if missing_in_child:
            print("  sample:", _topn(sorted(missing_in_child)))

        print(f"Child groups with UNKNOWN parent: {len(orphans_in_child)}"
              + (f" | child-pop-sum≈{child_orphan_total:,.0f}" if child_orphan_total is not None else ""))
        if orphans_in_child:
            print("  sample:", _topn(sorted(orphans_in_child)))
        print("="*88)

    # --- 1) Force ID dtypes on load (prevents mixed-type warnings later) -------------
    # You can keep your existing reads; just ensure these casts after reading.
    df10["GITTER_ID_10km"] = _norm_id(df10["GITTER_ID_10km"])
    df1["GITTER_ID_10km"]  = _norm_id(df1["GITTER_ID_10km"])
    df1["GITTER_ID_1km"]   = _norm_id(df1["GITTER_ID_1km"])
    df100["GITTER_ID_1km"] = _norm_id(df100["GITTER_ID_1km"])

    # --- 2) Run the two coverage checks ---------------------------------------------
    # 10km -> 1km
    _report_link(
        parent_df=df10,
        child_df=df1,
        parent_key="GITTER_ID_10km",
        child_parent_key="GITTER_ID_10km",
        parent_total_col="POP_TOTAL_10km_adj",  # if unavailable, set to None
        child_total_col="POP_TOTAL_1km",        # raw at load; before adj
        tag="10km → 1km"
    )

    # 1km -> 100m
    _report_link(
        parent_df=df1,
        child_df=df100,
        parent_key="GITTER_ID_1km",
        child_parent_key="GITTER_ID_1km",
        parent_total_col="POP_TOTAL_1km",       # raw at load; before adj
        child_total_col="POP_TOTAL_100m",
        tag="1km → 100m"
    )
# --------------------------------------------------------------------------------------------------------------------
    # ------------------------------------------------------------------------------
    # Tag linkage BEFORE running 1→100, but AFTER loading (10→1 runs normally)
    # ------------------------------------------------------------------------------
    def _norm_id(s: pd.Series) -> pd.Series:
        return s.astype(str).str.strip()

    p_1km = set(_norm_id(df1["GITTER_ID_1km"]).unique())
    c_100 = set(_norm_id(df100["GITTER_ID_1km"]).unique())

    missing_in_child = p_1km - c_100   # 1km parents that have NO 100m children
    orphans_in_child = c_100 - p_1km   # 100m rows with UNKNOWN 1km parent

    # Tag now so 1km output carries the info
    df1["has_no_children"] = _norm_id(df1["GITTER_ID_1km"]).isin(missing_in_child)
    df100["is_orphan"]     = _norm_id(df100["GITTER_ID_1km"]).isin(orphans_in_child)

    # ------------------------------------------------------------------------------
    # 10km → 1km (runs for ALL 1km rows)
    # ------------------------------------------------------------------------------
    make_child_totals_adj(
        parent_df=df10,
        child_df=df1,
        parent_id_col="GITTER_ID_10km",
        child_parent_id_col="GITTER_ID_10km",
        parent_adj_col="POP_TOTAL_10km_adj",
        child_pop_col="POP_TOTAL_1km",
    )

    ages_1km = downscale_single_years(
        parent_df=df10,
        child_df=df1,
        parent_id_col="GITTER_ID_10km",
        child_parent_id_col="GITTER_ID_10km",
        child_pop_col="POP_TOTAL_1km_adj",
        parent_age_cols=age_cols(100),
        child_level_for_bins="1km",
        cfg=DownscaleConfig(
            top_age=100, alpha_infr=0.80, alpha_10y=0.85, alpha_5c=0.90,
            trust_threshold=100.0, inner_passes=10, outer_iters=2,
            tol_rel_row=2e-4, rake_tol=1e-10, rake_max_iter=50
        )
    )
    df1_with_ages = df1.join(ages_1km)

    # Save 1km INCLUDING the has_no_children flag
    df1_with_ages.to_pickle(OUT_1KM)
    df1_with_ages.to_csv(OUT_1KM_CSV)

    # df1_with_ages = pd.read_pickle(OUT_1KM)
    # ------------------------------------------------------------------------------
    # 1km → 100m ONLY on clean linkage; propagate back to full 100m
    # ------------------------------------------------------------------------------
    df1_par_ok     = df1_with_ages.loc[~df1_with_ages["has_no_children"]].copy()
    df100_child_ok = df100.loc[~df100["is_orphan"]].copy()

    # # Filter both on 'CRS3035RES1000mN2791000E4481000' for debug
    # df1_par_ok = df1_par_ok[df1_par_ok['GITTER_ID_1km'] == 'CRS3035RES1000mN2791000E4481000']
    # df100_child_ok = df100_child_ok[df100_child_ok['GITTER_ID_1km'] == 'CRS3035RES1000mN2791000E4481000']

    # Totals adjustment on clean subset
    make_child_totals_adj(
        parent_df=df1_par_ok,
        child_df=df100_child_ok,
        parent_id_col="GITTER_ID_1km",
        child_parent_id_col="GITTER_ID_1km",
        parent_adj_col="POP_TOTAL_1km_adj",
        child_pop_col="POP_TOTAL_100m",   # writes POP_TOTAL_100m_adj on df100_child_ok
    )

    # Start with original totals; overwrite only adjusted clean rows
    df100["POP_TOTAL_100m_adj"] = df100["POP_TOTAL_100m"]
    df100.loc[df100_child_ok.index, "POP_TOTAL_100m_adj"] = df100_child_ok["POP_TOTAL_100m_adj"].values

    # Downscale ages on clean subset
    ages_100m_ok = downscale_single_years(
        parent_df=df1_par_ok,
        child_df=df100_child_ok,
        parent_id_col="GITTER_ID_1km",
        child_parent_id_col="GITTER_ID_1km",
        child_pop_col="POP_TOTAL_100m_adj",
        parent_age_cols=age_cols(100),
        child_level_for_bins="100m",
        cfg=DownscaleConfig(
            top_age=100, alpha_infr=0.80, alpha_10y=0.85, alpha_5c=0.90,
            trust_threshold=100.0, inner_passes=10, outer_iters=2,
            tol_rel_row=2e-4, rake_tol=1e-10, rake_max_iter=50
        )
    )

    # Merge ages back: clean rows get AGE_*; orphans keep NaN (or fill 0.0 if desired)
    df100_with_ages = df100.join(ages_100m_ok)
    # Optional: zero-fill orphans’ AGE_* to avoid NaNs downstream
    # df100_with_ages.loc[df100_with_ages["is_orphan"], "AGE_0":"AGE_100"] = 0.0

    # Save 100m INCLUDING is_orphan
    df100_with_ages.to_pickle(OUT_100M)
    df100_with_ages.to_csv(OUT_100M_CSV)

    # ------------------------------------------------------------------------------
    # Sanity (1km all; 100m ignore orphans)
    # ------------------------------------------------------------------------------
    def _echo(df_child: pd.DataFrame, pop_col: str, tag: str, mask=None):
        if mask is None:
            mask = pd.Series(True, index=df_child.index)
        calc = df_child.loc[mask, "AGE_0":"AGE_100"].sum(axis=1).to_numpy(float)
        given = pd.to_numeric(df_child.loc[mask, pop_col], errors="coerce").fillna(0.0).to_numpy(float)
        rel = np.abs(calc - given) / np.maximum(given, 1.0)
        print(f"[{tag}] rows checked={int(mask.sum()):,} | max row rel.err = {rel.max():.3e}, mean = {rel.mean():.3e}")

    _echo(df1_with_ages,   "POP_TOTAL_1km_adj",  "1km (adj)")
    _echo(df100_with_ages, "POP_TOTAL_100m_adj", "100m (adj, non-orphans only)", mask=~df100_with_ages["is_orphan"])

    # Check a parent block consistency (example for one 10km id)
    # pick a pid:
    if len(df1_with_ages):
        pid = df1_with_ages["GITTER_ID_10km"].iloc[3]
        # cols per age at 10km:
        P = df10.loc[df10["GITTER_ID_10km"] == pid, "AGE_0":"AGE_100"].sum(axis=0).to_numpy(float)
        C = df1_with_ages.loc[df1_with_ages["GITTER_ID_10km"] == pid, "AGE_0":"AGE_100"].sum(axis=0).to_numpy(float)
        print("[10km->1km] per-age max abs diff for one parent:", float(np.abs(P - C).max()))



[COVERAGE] 10km → 1km
----------------------------------------------------------------------------------------
Unique parent IDs: 3824
Unique child parent-IDs: 3824
Parents with NO children: 0 | parent-pop-sum≈0
Child groups with UNKNOWN parent: 0 | child-pop-sum≈0

[COVERAGE] 1km → 100m
----------------------------------------------------------------------------------------
Unique parent IDs: 212758
Unique child parent-IDs: 211486
Parents with NO children: 1314 | parent-pop-sum≈2,748
  sample: ['CRS3035RES1000mN2711000E4321000', 'CRS3035RES1000mN2715000E4352000', 'CRS3035RES1000mN2723000E4399000', 'CRS3035RES1000mN2725000E4285000', 'CRS3035RES1000mN2725000E4372000', 'CRS3035RES1000mN2726000E4472000', 'CRS3035RES1000mN2727000E4412000', 'CRS3035RES1000mN2727000E4418000', 'CRS3035RES1000mN2729000E4461000', 'CRS3035RES1000mN2730000E4376000', 'CRS3035RES1000mN2730000E4437000', 'CRS3035RES1000mN2731000E4404000', 'CRS3035RES1000mN2732000E4225000', 'CRS3035RES1000mN2732000E4428000', 'CRS3035

  0%|          | 585/211444 [00:18<1:50:42, 31.74it/s]


KeyboardInterrupt: 

In [8]:
df1["scale"].describe()

count    212758.000000
mean          1.000183
std           0.004055
min           0.818182
25%           0.999581
50%           1.000043
75%           1.000638
max           2.333333
Name: scale, dtype: float64

In [10]:
# Some smoke tests

import numpy as np
import pandas as pd

AGE_COLS = [c for c in df10.columns if c.startswith("AGE_")]  # AGE_0..AGE_100

# ---- generic helpers ----------------------------------------------------------
def _row_total_errors(df, pop_col, mask=None, age_cols=AGE_COLS):
    m = mask if mask is not None else pd.Series(True, index=df.index)
    calc = df.loc[m, age_cols].sum(axis=1).to_numpy(float)
    given = pd.to_numeric(df.loc[m, pop_col], errors="coerce").fillna(0.0).to_numpy(float)
    err_abs = calc - given
    err_rel = np.divide(np.abs(err_abs), np.maximum(given, 1.0))
    return err_abs, err_rel

def _assert_nonneg_and_finite(df, cols, tag):
    x = df[cols].to_numpy(dtype=float)
    assert np.isfinite(x).all(), f"[{tag}] found NaN/inf"
    assert (x >= -1e-9).all(), f"[{tag}] found negatives"

def _residual_report(err_abs, err_rel, tag):
    print(f"[{tag}] rows={err_abs.size:,} | "
          f"abs.max={np.max(np.abs(err_abs)):.6f} | "
          f"rel.max={np.max(err_rel):.3e} | rel.p99={np.quantile(err_rel,0.99):.3e} | rel.mean={np.mean(err_rel):.3e}")

# ---- 1) basic hygiene ---------------------------------------------------------
def check_hygiene():
    _assert_nonneg_and_finite(df10, AGE_COLS + ["POP_TOTAL_10km_adj"], "10km ages/totals")
    _assert_nonneg_and_finite(df1_with_ages, AGE_COLS + ["POP_TOTAL_1km_adj"], "1km ages/totals")
    _assert_nonneg_and_finite(df100_with_ages.loc[~df100_with_ages["is_orphan"]],
                              AGE_COLS + ["POP_TOTAL_100m_adj"], "100m ages/totals (non-orphans)")
    print("[HYGIENE] nonneg + finite: OK")

# ---- 2) per-row hard totals ---------------------------------------------------
def check_row_totals():
    for df, pop, tag, mask in [
        (df1_with_ages,   "POP_TOTAL_1km_adj",  "ROW 1km",       None),
        (df100_with_ages, "POP_TOTAL_100m_adj", "ROW 100m clean", ~df100_with_ages["is_orphan"]),
    ]:
        ea, er = _row_total_errors(df, pop, mask)
        _residual_report(ea, er, tag)
        assert np.max(er) <= 2e-4 + 1e-12, f"[{tag}] row rel.err exceeds 0.02%"

# ---- 3) parent per-age hard margins ------------------------------------------
def _group_age_sums(child, child_key, parent, parent_key, tag, child_mask=None, atol=1e-5):
    m = child_mask if child_mask is not None else pd.Series(True, index=child.index)
    C = (child.loc[m, [child_key]+AGE_COLS]
               .groupby(child_key, sort=False)[AGE_COLS].sum())
    P = (parent[[parent_key]+AGE_COLS]
               .groupby(parent_key, sort=False)[AGE_COLS].sum())
    # align
    C, P = C.align(P, join="inner", axis=0)
    diff = (C - P).to_numpy(float)
    mx = float(np.max(np.abs(diff))) if diff.size else 0.0
    print(f"[{tag}] parents checked={len(C):,} | per-age max|diff|={mx:.6f}")

def check_parent_margins():
    _group_age_sums(df1_with_ages, "GITTER_ID_10km", df10, "GITTER_ID_10km",
                    "10km → 1km per-age", atol=1e-3)
    _group_age_sums(df100_with_ages, "GITTER_ID_1km", df1_with_ages, "GITTER_ID_1km",
                    "1km → 100m per-age (clean)", child_mask=~df100_with_ages["is_orphan"], atol=1e-2)

# ---- 4) cross-level total aggregation (sanity, not hard) ----------------------
def check_crosslevel_totals():
    # 10km totals vs sum(1km adj)
    t10 = df10.set_index("GITTER_ID_10km")["POP_TOTAL_10km_adj"].astype(float)
    s1  = df1_with_ages.groupby("GITTER_ID_10km")["POP_TOTAL_1km_adj"].sum().astype(float)
    t10, s1 = t10.align(s1, join="inner")
    d10 = (s1 - t10).abs()
    print(f"[SUM] 10km vs Σ1km | max diff={float(d10.max()):.6f} | mean={float(d10.mean()):.6f}")

    # 1km totals vs sum(100m adj) on clean parents only
    clean_1km = df1_with_ages.loc[~df1_with_ages["has_no_children"]]
    t1  = clean_1km.set_index("GITTER_ID_1km")["POP_TOTAL_1km_adj"].astype(float)
    s100 = (df100_with_ages.loc[~df100_with_ages["is_orphan"]]
                           .groupby("GITTER_ID_1km")["POP_TOTAL_100m_adj"].sum().astype(float))
    t1, s100 = t1.align(s100, join="inner")
    d1 = (s100 - t1).abs()
    print(f"[SUM] 1km (clean) vs Σ100m | max diff={float(d1.max()):.6f} | mean={float(d1.mean()):.6f}")

# ---- 5) light diagnostics (optional quick smoke tests) ------------------------
def check_light_diags():
    # entropy of within-row age shares shouldn’t collapse at low n
    def _entropy(df, pop_col, mask=None):
        m = mask if mask is not None else pd.Series(True, index=df.index)
        X = df.loc[m, AGE_COLS].to_numpy(float)
        rs = np.maximum(X.sum(axis=1, keepdims=True), 1e-12)
        P = X / rs
        ent = -np.nansum(P * np.log(np.maximum(P, 1e-12)), axis=1)
        n = pd.to_numeric(df.loc[m, pop_col], errors="coerce").fillna(0.0).to_numpy(float)
        return n, ent
    for df, pop, tag, mask in [
        (df1_with_ages,   "POP_TOTAL_1km_adj",  "ENT 1km",       None),
        (df100_with_ages, "POP_TOTAL_100m_adj", "ENT 100m clean", ~df100_with_ages["is_orphan"]),
    ]:
        n, ent = _entropy(df, pop, mask)
        print(f"[{tag}] entropy: median={np.median(ent):.3f} | p10={np.quantile(ent,0.10):.3f} | "
              f"corr(n,ent)≈{np.corrcoef(n, ent)[0,1]:.3f}")

# ---- run all ------------------------------------------------------------------
print("\n=== SANITY CHECKS START ===")
df10 = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\merged\df10_with_single_years.pickle").reset_index(drop=False)
df100_with_ages = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\merged\df100_with_single_years.pickle").reset_index(drop=False)
df1_with_ages = pd.read_pickle(r"C:\Users\petre\PycharmProjects\cleancensus\merged\df1_with_single_years.pickle").reset_index(drop=False)
check_hygiene()
check_row_totals()
check_parent_margins()
check_crosslevel_totals()
check_light_diags()
print("=== SANITY CHECKS DONE ===\n")



=== SANITY CHECKS START ===
[HYGIENE] nonneg + finite: OK
[ROW 1km] rows=212,758 | abs.max=0.000000 | rel.max=1.667e-15 | rel.p99=8.613e-16 | rel.mean=2.512e-16
[ROW 100m clean] rows=3,148,437 | abs.max=0.000000 | rel.max=2.103e-15 | rel.p99=8.831e-16 | rel.mean=2.555e-16
[10km → 1km per-age] parents checked=3,824 | per-age max|diff|=0.000000
[1km → 100m per-age (clean)] parents checked=211,444 | per-age max|diff|=0.261530
[SUM] 10km vs Σ1km | max diff=0.000000 | mean=0.000000
[SUM] 1km (clean) vs Σ100m | max diff=0.000000 | mean=0.000000
[ENT 1km] entropy: median=4.330 | p10=4.127 | corr(n,ent)≈0.143
[ENT 100m clean] entropy: median=4.281 | p10=4.014 | corr(n,ent)≈0.166
=== SANITY CHECKS DONE ===

